# 🚀 Advanced Mid-Retrieval: Orchestrating Techniques

## What this notebook covers

**3 advanced techniques** that intelligently **orchestrate** the basic methods:

1. **Multi-Query Retrieval** - Generate query variants for broader coverage
2.  **Query Routing** - Auto-select the right technique per query type
3. **Adaptive Retrieval** - Auto-configure based on query complexity

Plus: **How to combine techniques** for production RAG systems!

---

## The Key Idea

```
        Individual Tools
Dense, Hybrid, Reranking, Parent-Child
               ↓
        Smart Orchestration  
When to use which tool? How to combine them?
               ↓
       Production RAG System
```

Let's build your expertise! 🎯

## Setup: Connect to Vector Database

In [1]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

from retrieval_playground.utils import constants
from retrieval_playground.utils.model_manager import model_manager
from retrieval_playground.src.mid_retrieval.multi_query_hybrid import MultiQueryHybrid
from retrieval_playground.src.mid_retrieval.adaptive_retrieval import AdaptiveRetriever
from retrieval_playground.src.mid_retrieval.route_driven_retrieval import RouteRetriever

import warnings
warnings.filterwarnings('ignore')

# Connect to Qdrant
qdrant_client = QdrantClient(url=constants.QDRANT_URL, api_key=constants.QDRANT_KEY)
embeddings = model_manager.get_embeddings()

# Create vector store (using recursive_character from 2A)
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="recursive_character",
    embedding=embeddings
)

print("✅ Setup complete!")
print("   Collection: recursive_character")
print("   Ready to explore advanced orchestration!")

2026-07-04 20:50:09.870 INFO model_manager - _initialize_models: 🔄 ModelManager: Initializing shared AI models...


2026-07-04 20:50:09 - rp_logger - INFO - model_manager.py:65 - _initialize_models() - 🔄 ModelManager: Initializing shared AI models...


2026-07-04 20:50:09.937 INFO model_manager - _initialize_models: ✅ ModelManager: Shared AI models initialized successfully


2026-07-04 20:50:09 - rp_logger - INFO - model_manager.py:81 - _initialize_models() - ✅ ModelManager: Shared AI models initialized successfully
2026-07-04 20:50:09 WARNING semantic_router No index provided. Using default LocalIndex.
2026-07-04 20:50:14 WARNING semantic_router No config is written for LocalIndex.


✅ Setup complete!
   Collection: recursive_character
   Ready to explore advanced orchestration!


---

# 🔄 Technique 5: Multi-Query Retrieval

## The Problem: Vocabulary Mismatch

Even with semantic search, a **single query** can miss relevant documents:

**Example:**
- **Your query:** "How do neural networks learn?"
- **Missed docs using:** "training process", "backpropagation", "gradient descent", "optimization algorithms"

Why? Dense search creates ONE vector from ONE phrasing. Different vocabulary = different embeddings.

---

## The Solution: Multi-Query Pipeline

Our implementation combines **FOUR stages** for maximum quality:

```
Original Query: "How do neural networks learn?"
                            ↓
        ┌─────────────────────────────────────────┐
        │    Stage 1: Generate Variants (LLM)     │
        └─────────────────────────────────────────┘
                            ↓
    ┌────────────────┬────────────────┬────────────────┐
    ↓                ↓                ↓                ↓
Variant 1:     Variant 2:       Variant 3:       Original:
"Neural net    "How does        "Learning        "How do NNs
training       backprop         mechanisms in     learn?"
process"       work in NNs?"    deep learning"
    ↓                ↓                ↓                ↓
        ┌─────────────────────────────────────────┐
        │    Stage 2: Hybrid Search (BM25+Dense)  │
        │     Run for EACH variant (4 searches)   │
        └─────────────────────────────────────────┘
    ↓                ↓                ↓                ↓
    └────────────────┴────────────────┴────────────────┘
                            ↓
       ┌─────────────────────────────────────────┐
       │           Stage 3: RRF Fusion           │
       │         Merge all 4 result sets         │
       └─────────────────────────────────────────┘
                            ↓
       ┌─────────────────────────────────────────┐
       │   Stage 4: Reranking (Cross-Encoder)    │
       │    Top n candidates → Top 5 precise     │
       └─────────────────────────────────────────┘
                          ↓
                     Final Results
                  (Maximum Quality! 🎯)
```

**This combines THREE techniques:**
1. Hybrid Search (BM25 + Dense + RRF)
2. Reranking (cross-encoder precision)
3. Multi-Query orchestration (NEW in 2B!)

---

## How It Works: Step by Step

**Stage 1: Generate Variants (Using LLM)**
```
Prompt to LLM: "Generate 3 alternative phrasings of: {query}"
               "Use different vocabulary but keep the same meaning"

Output: 3 query variants with diverse vocabulary
```

**Stage 2: Hybrid Search for Each Variant**
```
4 searches total (each is BM25 + Dense + RRF):
  - Original query → Hybrid results set 1
  - Variant 1 → Hybrid results set 2
  - Variant 2 → Hybrid results set 3
  - Variant 3 → Hybrid results set 4
```

**Stage 3: Merge with RRF**
```
Same RRF formula from Hybrid Search:

RRF_score = 1/(60 + rank_v1) + 1/(60 + rank_v2) + 
            1/(60 + rank_v3) + 1/(60 + rank_original)

Documents appearing in MULTIPLE variant results score higher!
```

**Stage 4: Rerank with Cross-Encoder**
```
Take top n candidates from RRF fusion
  ↓
Pass to cross-encoder
  ↓
Get precise relevance scores (0-1)
  ↓
Return top 5 sorted by rerank_score
```

---

## Understanding the Scores

**After Stage 3 (RRF):** Documents have `rrf_score` (0-0.035 range)  
**After Stage 4 (Reranking):** Documents have `rerank_score` (0-1 range) ← **FINAL SORTING**

**Final results are sorted by `rerank_score`**, not `rrf_score`!

Cross-encoder gives the final word on relevance!

---

## When Multi-Query Pipeline Shines

**✅ Perfect for:**
- Complex questions with multiple valid phrasings
- Technical domains where terminology varies ("ML training" vs "model optimization")
- Broad exploratory questions ("How does X work?")
- Research queries needing comprehensive coverage
- High-stakes applications (medical, legal, financial)

**❌ Not ideal for:**
- Simple factual lookups ("What is BERT?") - overkill
- Queries with very specific unique terms 
- When speed is critical (4x searches + reranking = ~400-500ms)
- Low-cost deployments (4x search cost)

## Try It: See the Variants

Let's first see what query variants look like:

In [2]:
from retrieval_playground.src.pre_retrieval.query_rephrasing import expand_query

# Example query
query = "How do scientific agents plan and decompose complex research tasks?"

# Generate variants
variants = expand_query(query, num_variants=3)

print("🔄 MULTI-QUERY VARIANT GENERATION")
print("=" * 80)
print(f"\n📝 Original Query:")
print(f"   {query}\n")

print("🔄 Generated Variants:")
for i, variant in enumerate(variants, 1):
    print(f"   {i}. {variant}")

print("\n💡 Notice how each variant uses different vocabulary!")
print("=" * 80)

2026-07-04 20:50:16 - google_genai.models - INFO - models.py:6472 - generate_content() - AFC is enabled with max remote calls: 10.


🔄 MULTI-QUERY VARIANT GENERATION

📝 Original Query:
   How do scientific agents plan and decompose complex research tasks?

🔄 Generated Variants:
   1. Methods used by autonomous AI for breaking down and strategizing complex scientific problems.
   2. Task decomposition and planning strategies for autonomous agents in scientific discovery.
   3. What is the process for an AI research assistant to segment and schedule a multi-step investigation?

💡 Notice how each variant uses different vocabulary!


## Try It: Multi-Query in Action

Now let's use Multi-Query with Hybrid Search:

In [3]:
# Initialize multi-query retriever (uses Hybrid Search + Reranking internally)
mqh = MultiQueryHybrid(collection_name="hybrid")

# Search using query variants
query = "How do scientific agents plan and decompose complex research tasks?"
results = mqh.search(query, k=5)

print("🔄 MULTI-QUERY HYBRID RESULTS")
print("=" * 80)
print(f"Query: '{query}'\n")
print("Full Pipeline (4 stages):")
print("  1. Generated 3 query variants (different vocabulary)")
print("  2. Ran Hybrid Search (BM25 + Dense) for each variant")
print("  3. Merged all results using RRF")
print("  4. Reranked top 100 with cross-encoder (returns top 5)\n")

for i, doc in enumerate(results, 1):
    # After reranking, results have rerank_score (sorted by this!)
    rerank_score = doc.metadata.get('rerank_score', 0)
    print(f"{i}. Rerank Score: {rerank_score:.4f}")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:120]}...\n")

print("=" * 80)
print("💡 This is the FULL pipeline combining THREE techniques:")
print("   - Multi-Query (variant generation)")
print("   - Hybrid Search (BM25 + Dense + RRF)")
print("   - Reranking (cross-encoder precision)")
print("   Result: Maximum quality! 🎯")
print("=" * 80)

mqh.close()

✅ Hybrid retriever initialized (collection: 'hybrid')
2026-07-04 20:50:28.093 INFO model_manager - get_reranker: ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2) [cache: /Users/maharora/Documents/Other/github/retrieval-playground/data/models/flashrank]


2026-07-04 20:50:28 - rp_logger - INFO - model_manager.py:150 - get_reranker() - ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2) [cache: /Users/maharora/Documents/Other/github/retrieval-playground/data/models/flashrank]


2026-07-04 20:50:28.094 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 100, top_n: 5)


2026-07-04 20:50:28 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 100, top_n: 5)
2026-07-04 20:50:28 - google_genai.models - INFO - models.py:6472 - generate_content() - AFC is enabled with max remote calls: 10.


✅ Multi-query hybrid initialized (3 variants, FlashRank reranker)
🔄 MULTI-QUERY HYBRID RESULTS
Query: 'How do scientific agents plan and decompose complex research tasks?'

Full Pipeline (4 stages):
  1. Generated 3 query variants (different vocabulary)
  2. Ran Hybrid Search (BM25 + Dense) for each variant
  3. Merged all results using RRF
  4. Reranked top 100 with cross-encoder (returns top 5)

1. Rerank Score: 0.9954
   Source: /Users/maharora/Documents/Other/github/retrieval-playground/data/workshop_data/2025_Scientific_Intelligence_Survey.pdf
   Preview: 2 Architecture
The architecture of LLM-based scientific agents is
designed to enable iterative, context-aware process-
i...

2. Rerank Score: 0.9932
   Source: /Users/maharora/Documents/Other/github/retrieval-playground/data/workshop_data/2025_Scientific_Intelligence_Survey.pdf
   Preview: sequences that orchestrate tool invocations, mem-
ory operations, and verification steps. In au-
tonomous scientific dis...

3. Rerank Score: 

## Experiment: Compare Single vs Multi-Query

Let's see the difference between single query and multi-query:

In [4]:
import time
from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever

query = "What verification methods ensure reproducibility in scientific experiments?"

# Single query (baseline Hybrid) - measure time INCLUDING initialization
start_single = time.time()
hybrid = HybridRetriever(collection_name="hybrid")
single_results = hybrid.search(query, k=3)
time_single = (time.time() - start_single) * 1000  # Convert to ms

# Multi-query (includes reranking!) - measure time INCLUDING initialization
start_multi = time.time()
mqh = MultiQueryHybrid(collection_name="hybrid")
multi_results = mqh.search(query, k=3)
time_multi = (time.time() - start_multi) * 1000  # Convert to ms

print("=" * 80)
print("SINGLE QUERY vs MULTI-QUERY COMPARISON")
print("=" * 80)
print(f"\nQuery: '{query}'\n")

print(f"📊 SINGLE QUERY (Hybrid Search only) - {time_single:.0f}ms:")
for i, doc in enumerate(single_results, 1):
    score = doc.metadata.get('rrf_score', 0)
    print(f"  {i}. RRF: {score:.4f} | {doc.page_content[:70]}...")

print(f"\n🔄 MULTI-QUERY (4 variants + Hybrid + Reranking) - {time_multi:.0f}ms:")
for i, doc in enumerate(multi_results, 1):
    # Multi-query returns reranked results, so use rerank_score
    score = doc.metadata.get('rerank_score', 0)
    print(f"  {i}. Rerank: {score:.4f} | {doc.page_content[:70]}...")

print("\n💡 Multi-query pipeline finds more relevant results!")
print("   Trade-offs:")
print(f"   - Single: {time_single:.0f}ms, good quality")
print(f"   - Multi-query: {time_multi:.0f}ms, best quality + sorted by reranking")
print(f"   - Speedup: {time_multi/time_single:.1f}x slower")
print("=" * 80)

hybrid.close()
mqh.close()

✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Hybrid retriever initialized (collection: 'hybrid')
2026-07-04 20:50:51.859 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 100, top_n: 5)


2026-07-04 20:50:51 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 100, top_n: 5)
2026-07-04 20:50:51 - google_genai.models - INFO - models.py:6472 - generate_content() - AFC is enabled with max remote calls: 10.


✅ Multi-query hybrid initialized (3 variants, FlashRank reranker)
SINGLE QUERY vs MULTI-QUERY COMPARISON

Query: 'What verification methods ensure reproducibility in scientific experiments?'

📊 SINGLE QUERY (Hybrid Search only) - 2537ms:
  1. RRF: 0.0313 | hypotheses, experimental designs, analytical pro-
cedures, and scienti...
  2. RRF: 0.0308 | evaluating whether proposed hypotheses explore
sufficiently diverse re...
  3. RRF: 0.0302 | Verifier
V1. Self-Correction
/ Reflective
Verification
AI Scientist (L...

🔄 MULTI-QUERY (4 variants + Hybrid + Reranking) - 21294ms:
  1. Rerank: 0.9520 | hypotheses, experimental designs, analytical pro-
cedures, and scienti...
  2. Rerank: 0.9065 | tor agents. Chen and Madisetti (2024) adds that
multi-agent communicat...
  3. Rerank: 0.7967 | 5.7 Intellectual Property and Research
Integrity
LLM integration in re...

💡 Multi-query pipeline finds more relevant results!
   Trade-offs:
   - Single: 2537ms, good quality
   - Multi-query: 21294ms, best q

---

# 🎯 Technique 6: Query Routing

## The Problem: One Size Doesn't Fit All

Not all queries need the same retrieval approach:

**Examples:**
- **"Hello!"** → No retrieval needed (greeting)
- **"What is BERT?"** → Simple Dense Search (factual lookup)
- **"Compare BERT and GPT-3 architectures"** → Multi-Query (needs breadth)
- **"How do transformers handle long sequences?"** → Hybrid + Reranking (complex)

Using Multi-Query + Reranking for "Hello!" wastes resources.  
Using Dense-only for complex comparisons misses relevant content.

---

## The Solution: Semantic Routing

**Automatically detect query type** and select the appropriate method:

```
User Query
     ↓
Semantic Router (analyzes intent)
     ↓
  ┌──────────┬──────────┬──────────┐
  ↓          ↓          ↓          ↓         
Greeting  Factual   Comparison  Complex/Technical   
  ↓          ↓          ↓          ↓          
No         Dense     Multi-     Hybrid+   
Retrieval  Search    Query      Rerank  
```

---

## How It Works: Semantic Routing

**Step 1: Define Routes**
```python
routes = [
    Route(
        name="greeting",
        utterances=["hello", "hi", "hey", "good morning"],
        requires_retrieval=False
    ),
    Route(
        name="factual",
        utterances=["what is", "define", "explain briefly"],
        method="dense"
    ),
    Route(
        name="comparison",
        utterances=["compare", "difference between", "X vs Y"],
        method="multi_query"
    ),
    Route(
        name="complex",
        utterances=["how does X work", "explain the process"],
        method="hybrid_rerank"
    )
]
```

**Step 2: Route Query**
```
Query: "Compare BERT and GPT-3"
  ↓
Router embeds query and utterances
  ↓
Finds closest match: "comparison" route
  ↓
Returns: method="multi_query"
  ↓
Executes Multi-Query Hybrid
```

**Step 3: Execute Method**
```python
if route.requires_retrieval == False:
    return []  # No retrieval needed
elif route.method == "dense":
    return dense_search(query)
elif route.method == "multi_query":
    return multi_query_search(query)
elif route.method == "hybrid_rerank":
    return hybrid_search(query) + rerank()
```

---

## Benefits of Routing

**1. Efficiency** 
- Greetings skip retrieval entirely
- Simple queries use fast Dense Search
- Complex queries get full pipeline

**2. Quality** 
- Method matches query complexity
- Comparisons get breadth (Multi-Query)
- Complex questions get precision (Reranking)

**3. Cost Savings**
- Don't waste resources on simple queries

---

## When to Use Routing

**✅ Perfect for:**
- Production chatbots with varied query types
- Applications with mix of simple/complex queries
- Cost-sensitive deployments
- Multi-domain systems

**❌ Not needed for:**
- Single query type (all factual or all complex)
- Research/prototyping (just use best method)
- Very small scale (routing overhead not worth it)

## Try It: See Routing in Action

In [5]:
# Initialize route-driven retriever
route_retriever = RouteRetriever(collection_name="hybrid")

print("🎯 QUERY ROUTING - Example 1: Greeting")
print("=" * 80)

query = "Hello!"
print(f"\n📝 Query: '{query}'")

# Get route info first
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Requires Retrieval: {route_info['route']['requires_retrieval']}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\n🔄 Processing:")

# Execute search
results = route_retriever.search(query)
print(f"\n📊 Results: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100]}...")

print("=" * 80)

✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Route-driven retriever initialized
🎯 QUERY ROUTING - Example 1: Greeting

📝 Query: 'Hello!'

📍 Route Classification:
   Detected Type: GREETINGS
   Requires Retrieval: False
   Complexity: simple

🔄 Processing:

📊 Results: 0 documents


In [6]:
print("🎯 QUERY ROUTING - Example 2: Factual Question")
print("=" * 80)

query = "What is BERT?"
print(f"\n📝 Query: '{query}'")

# Get route info
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Selected Method: {route_info['final_retrieval_method'].upper()}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\n🔄 Processing:")

# Execute search
results = route_retriever.search(query)

print(f"\n📊 Results: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100]}...")

print("=" * 80)

🎯 QUERY ROUTING - Example 2: Factual Question

📝 Query: 'What is BERT?'

📍 Route Classification:
   Detected Type: FACTUAL
   Selected Method: HYBRID_SEARCH
   Complexity: simple

🔄 Processing:

📊 Results: 5 documents

   Top Result:
   Sabater, J. Nicolas, C. Peubey, R. Radu, D. Scheperset al., “The
era5 global reanalysis,”Quarterly J...


In [7]:
print("🎯 QUERY ROUTING - Example 3: Comparison Question")
print("=" * 80)

query = "Compare BERT and GPT-3 architectures"
print(f"\n📝 Query: '{query}'")

# Get route info
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Selected Method: {route_info['final_retrieval_method'].upper()}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\n🔄 Processing:")
print(f"   Stage 1: Generate 3 query variants (LLM)")

# Generate variants to show
from retrieval_playground.src.pre_retrieval.query_rephrasing import expand_query
variants = expand_query(query, num_variants=3)
for i, variant in enumerate(variants, 1):
    print(f"      Variant {i}: {variant[:60]}...")

# Execute search
results = route_retriever.search(query)

print(f"\n📊 Results: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100]}...")

print("=" * 80)

🎯 QUERY ROUTING - Example 3: Comparison Question

📝 Query: 'Compare BERT and GPT-3 architectures'


2026-07-04 20:51:16 - google_genai.models - INFO - models.py:6472 - generate_content() - AFC is enabled with max remote calls: 10.



📍 Route Classification:
   Detected Type: COMPARISON
   Selected Method: MULTI_QUERY
   Complexity: moderate

🔄 Processing:
   Stage 1: Generate 3 query variants (LLM)
      Variant 1: What are the key architectural differences between BERT and ...
      Variant 2: BERT vs GPT-3: a comparison of their underlying model design...
      Variant 3: A breakdown of the structural distinctions between the BERT ...


2026-07-04 20:51:33 - google_genai.models - INFO - models.py:6472 - generate_content() - AFC is enabled with max remote calls: 10.



📊 Results: 8 documents

   Top Result:
   and ChatGPT (Achiam et al. (2023); Hurst et al. (2024); OpenAI (2022)), leverage vast datasets
and s...


In [8]:
print("🎯 QUERY ROUTING - Example 4: Complex Question")
print("=" * 80)

query = "How do attention mechanisms work in transformers?"
print(f"\n📝 Query: '{query}'")

# Get route info
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Selected Method: {route_info['final_retrieval_method'].upper()}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\n🔄 Processing:")


# Execute search
results = route_retriever.search(query)

print(f"\n📊 Results: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100]}...")

print("\n" + "=" * 80)

route_retriever.close()

🎯 QUERY ROUTING - Example 4: Complex Question

📝 Query: 'How do attention mechanisms work in transformers?'

📍 Route Classification:
   Detected Type: ANALYTICAL
   Selected Method: HYBRID_SEARCH
   Complexity: complex

🔄 Processing:

📊 Results: 8 documents

   Top Result:
   z′
ℓ =MSA(LN(z ℓ−1)) +zℓ−1, ℓ=1, . . . ,L,
zℓ,=MLP
 
LN
 
z′
ℓ

+z ′
ℓ, ℓ=1, . . . ,L,
where MSA i...



---

# 🧠 Technique 7: Adaptive Retrieval

## The Problem: Fixed Parameters Waste Resources

In 2A, we used fixed parameters:
- `k=5` (always return 5 results)
- Always same method (Dense, Hybrid, etc.)
- Same threshold for all queries

But different queries have different needs:

**Simple query:**  
"What is BERT?" → Needs k=2, fast Dense Search

**Complex query:**  
"Compare BERT, GPT-3, and T5 architectures..." → Needs k=10, Hybrid + Reranking

**The waste:**  
Using k=10 for "What is BERT?" returns 8 irrelevant docs  
Using k=2 for complex comparison misses important content

---

## The Solution: Adaptive Configuration

**Automatically analyze query complexity** and adjust parameters:

```
              User Query
                  ↓
         Complexity Analyzer
                  ↓
  ┌──────────┬──────────┬──────────┐
  ↓          ↓          ↓          ↓
Simple   Moderate   Complex   Very Complex
  ↓          ↓          ↓          ↓
k=2        k=5        k=8        k=12
Dense      Hybrid     Hybrid+    Multi-Query+
                      Rerank     Rerank
```

---

## How It Works: Complexity Analysis

**Step 1: Analyze Query**
```python
def analyze_complexity(query):
    score = assign_complexity_score(query)
    
    # Classify
    if score <= 1: return 'simple'
    elif score <= 3: return 'moderate'
    elif score <= 5: return 'complex'
    else: return 'very_complex'
```

**Step 2: Configure Parameters**
```python
config_map = {
    'simple': {
        'k': 2,
        'method': 'dense',
        'use_reranking': False,
        'threshold': 0.7
    },
    'moderate': {
        'k': 5,
        'method': 'hybrid',
        'use_reranking': False,
        'threshold': 0.6
    },
    'complex': {
        'k': 8,
        'method': 'hybrid',
        'use_reranking': True,
        'threshold': 0.5
    },
    'very_complex': {
        'k': 12,
        'method': 'multi_query',
        'use_reranking': True,
        'threshold': 0.4
    }
}
```

**Step 3: Execute with Config**
```python
complexity = analyze_complexity(query)
config = config_map[complexity]
results = retrieve(query, **config)
```

---

## Benefits of Adaptive Retrieval

**1. Efficiency** 
- Simple queries use minimal resources (k=2, Dense), Complex queries get full power (k=12, Multi-Query + Rerank)

**2. Quality** 
- Simple queries avoid noise (fewer irrelevant results), Complex queries get breadth (more candidates)

**3. Automatic** 
- No manual tuning per query, Self-adjusting system

---

## When to Use Adaptive Retrieval

**✅ Perfect for:**
- Production systems with varied query complexity
- Unknown query patterns (can't predict difficulty)
- Cost-sensitive deployments
- "Set and forget" solutions

**❌ Not needed for:**
- Homogeneous query complexity (all simple or all complex)
- When you want explicit control
- Very small scale (complexity analysis overhead)

## Try It: See Adaptive Configuration

In [11]:
# Initialize adaptive retriever
adaptive = AdaptiveRetriever(collection_name="hybrid")

# Test with different complexity queries
queries = [
    "What is BERT?",
    "How do attention mechanisms work in transformers?",
    "Compare and analyze BERT, GPT-3, and T5 architectures, explaining their training methods and use cases"
]

print("🧠 ADAPTIVE RETRIEVAL DEMONSTRATION")
print("=" * 80)

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 80)
    
    # Search with adaptive configuration
    results = adaptive.search(query)
    
    print(f"   Results: {len(results)} documents")
    if results:
        print(f"   Top result: {results[0].page_content[:60]}...")
    print()

print("=" * 80)
print("💡 Notice how configuration adapts to query complexity:")
print("   - Simple: k=2-3, Dense Search (fast!)")
print("   - Moderate: k=5, Hybrid Search")
print("   - Complex: k=8-12, Multi-Query + Reranking (comprehensive!)")
print("=" * 80)

adaptive.close()

✅ Hybrid retriever initialized (collection: 'hybrid')
2026-07-04 20:58:53.664 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 8)


2026-07-04 20:58:53 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 8)


✅ Hybrid retriever initialized (collection: 'hybrid')
2026-07-04 20:58:55.113 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 100, top_n: 5)


2026-07-04 20:58:55 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 100, top_n: 5)


✅ Multi-query hybrid initialized (3 variants, FlashRank reranker)
✅ Adaptive retriever initialized
🧠 ADAPTIVE RETRIEVAL DEMONSTRATION

Query: 'What is BERT?'
--------------------------------------------------------------------------------


2026-07-04 20:58:56 - google_genai.models - INFO - models.py:6472 - generate_content() - AFC is enabled with max remote calls: 10.


   Results: 3 documents
   Top result: sentence-transformers. The complete schema with node/edge
de...


Query: 'How do attention mechanisms work in transformers?'
--------------------------------------------------------------------------------
   Results: 8 documents
   Top result: Cross-Attention Diffusion (Cross-Attn)51.A diffusion framewo...


Query: 'Compare and analyze BERT, GPT-3, and T5 architectures, explaining their training methods and use cases'
--------------------------------------------------------------------------------
   Results: 5 documents
   Top result: 112.2. COMPARISON OF PARALLEL ARCHITECTURES23
originally des...

💡 Notice how configuration adapts to query complexity:
   - Simple: k=2-3, Dense Search (fast!)
   - Moderate: k=5, Hybrid Search
   - Complex: k=8-12, Multi-Query + Reranking (comprehensive!)


## Experiment: Manual Override

You can override adaptive configuration when needed:

In [ ]:
adaptive = AdaptiveRetriever(collection_name="hybrid")

query = "What is BERT?"  # Simple query

print("🧪 MANUAL OVERRIDE EXPERIMENT")
print("=" * 80)

# 1. Default adaptive behavior
print("\n1. DEFAULT (Adaptive):")
results_adaptive = adaptive.search(query, verbose=True)
print(f"   Results: {len(results_adaptive)}")

# 2. Force complex configuration
print("\n2. OVERRIDE (Force complex config):")
custom_config = {
    "method": "multi_query",
    "k": 10,
    "use_reranking": True,
    "threshold": 0.3
}
results_override = adaptive.search(query, force_config=custom_config, verbose=True)
print(f"   Results: {len(results_override)}")

print("\n=" * 80)
print("💡 Adaptive chose k=2-3 (efficient for simple query)")
print("   Override used k=10 (overkill, but you have control!)")
print("=" * 80)

adaptive.close()

---

# 🚀 Combining Techniques: Building Production Pipelines

Now that you've learned all 7 techniques, let's see how to **combine** them effectively!

---

## The Power of Combination

**Individual techniques** from 2A and 2B each provide benefits:  
But **combining** them creates **multiplicative** improvements!

**Example gains:**
- Hybrid Search alone: +15-25% over Dense
- Reranking alone: +10-15% over Dense
- **Hybrid + Reranking: +30-40%** (not just 25%!) 🎯

Why? Each technique fixes different problems:
- Hybrid: Adds keyword matching (fixes vocabulary gap)
- Reranking: Adds query-document interaction (fixes shallow scoring)
- Multi-Query: Adds variant coverage (fixes single-phrasing bias)
- Adaptive: Adds efficiency (fixes resource waste)

---

## Common Production Pipelines

### Pipeline 1: Balanced Performance
**Best for:** General-purpose production RAG

```
Query → Routing → Hybrid Search → Reranking → Top-5
```

**Techniques:**
1. Routing (skip retrieval for greetings)
2. Hybrid Search (BM25 + Dense + RRF)
3. Reranking (FlashRank for speed)

**Performance:** +30-35% quality, ~150ms latency  
**Use when:** Good quality needed, moderate latency acceptable

---

### Pipeline 2: Maximum Quality
**Best for:** High-stakes applications (medical, legal, financial)

```
Query → Multi-Query → Hybrid Search → Reranking → Top-3
```

**Techniques:**
1. Multi-Query (generate 3 variants)
2. Hybrid Search for each variant (12 searches!)
3. RRF fusion across all results
4. Reranking (BGE or ms-marco for precision)

**Performance:** +45-60% quality, ~500ms latency  
**Use when:** Precision is critical, latency less important

---

### Pipeline 3: Speed Optimized
**Best for:** High-throughput, cost-sensitive applications

```
Query → Adaptive → Dense/Hybrid (adaptive) → Top-k (adaptive)
```

**Techniques:**
1. Adaptive Retrieval (auto-configure complexity)
2. Simple queries: Dense, k=2 (~20ms)
3. Complex queries: Hybrid, k=8 (~100ms)

**Performance:** +20-25% quality, ~50ms avg latency  
**Use when:** Speed/cost critical, acceptable quality loss

---

### Pipeline 4: Intelligent Adaptive
**Best for:** Unknown query patterns, set-and-forget systems

```
Query → Routing → Adaptive → Method (adaptive) → Top-k (adaptive)
```

**Techniques:**
1. Routing (filter greetings/chitchat)
2. Adaptive Retrieval:
   - Simple: Dense, k=2
   - Moderate: Hybrid, k=5
   - Complex: Multi-Query + Rerank, k=10

**Performance:** +30-50% quality (query-dependent), ~100ms avg  
**Use when:** Need automatic optimization, varied queries

---

## Choosing Your Pipeline

**Decision matrix:**

| Priority | Pipeline | Latency | Quality | Complexity |
|----------|----------|---------|---------|------------|
| **Speed** | Adaptive | ~50ms | +20-25% | Low |
| **Balance** | Routing + Hybrid + Rerank | ~150ms | +30-35% | Medium |
| **Quality** | Multi-Query + Hybrid + Rerank | ~500ms | +45-60% | High |
| **Smart** | Routing + Adaptive | ~100ms | +30-50% | Medium |

**Pro tip:** Start with **Balanced** (Pipeline 1), then optimize based on:
- If too slow → Switch to **Speed** (Pipeline 3)
- If quality insufficient → Upgrade to **Quality** (Pipeline 2)
- If varied queries → Switch to **Smart** (Pipeline 4)

## Try It: Compare Pipelines

Let's compare different pipeline configurations:

In [ ]:
import time
from retrieval_playground.src.mid_retrieval.reranking import Reranker

query = "How do attention mechanisms work in transformer neural networks?"

print("🚀 PIPELINE COMPARISON")
print("=" * 80)
print(f"Query: '{query}'\n")

# Pipeline 1: Dense Only (baseline from 2A)
print("1️⃣  BASELINE - Dense Only")
start = time.time()
results_dense = vector_store.similarity_search_with_score(query, k=5)
latency_dense = (time.time() - start) * 1000
top_score_dense = results_dense[0][1] if results_dense else 0
print(f"   Latency: {latency_dense:.0f}ms")
print(f"   Top score: {top_score_dense:.4f}")
print(f"   Preview: {results_dense[0][0].page_content[:60]}...\n")

# Pipeline 2: Hybrid Search
print("2️⃣  HYBRID - BM25 + Dense + RRF")
hybrid = HybridRetriever(collection_name="hybrid")
start = time.time()
results_hybrid = hybrid.search(query, k=5)
latency_hybrid = (time.time() - start) * 1000
top_score_hybrid = results_hybrid[0].metadata.get('rrf_score', 0) if results_hybrid else 0
print(f"   Latency: {latency_hybrid:.0f}ms")
print(f"   Top RRF score: {top_score_hybrid:.4f}")
print(f"   Preview: {results_hybrid[0].page_content[:60]}...\n")
hybrid.close()

# Pipeline 3: Hybrid + Reranking (Dense → Rerank)
print("3️⃣  BALANCED - Dense → Reranking")
reranker = Reranker(collection_name="recursive_character", top_k=20, top_n=5)
start = time.time()
results_rerank = reranker.retrieve(query)
latency_rerank = (time.time() - start) * 1000
top_score_rerank = results_rerank[0].metadata.get('rerank_score', 0) if results_rerank else 0
print(f"   Latency: {latency_rerank:.0f}ms")
print(f"   Top rerank score: {top_score_rerank:.4f}")
print(f"   Preview: {results_rerank[0].page_content[:60]}...\n")
reranker.close()

# Pipeline 4: Multi-Query Hybrid (Full pipeline with reranking)
print("4️⃣  QUALITY - Multi-Query + Hybrid + Reranking")
mqh = MultiQueryHybrid(collection_name="hybrid")
start = time.time()
results_mqh = mqh.search(query, k=5)
latency_mqh = (time.time() - start) * 1000
# Multi-query returns reranked results
top_score_mqh = results_mqh[0].metadata.get('rerank_score', 0) if results_mqh else 0
print(f"   Latency: {latency_mqh:.0f}ms")
print(f"   Top rerank score: {top_score_mqh:.4f}")
print(f"   Preview: {results_mqh[0].page_content[:60]}...\n")
mqh.close()

print("=" * 80)
print("📊 SUMMARY:")
print(f"   Baseline (Dense):         {latency_dense:.0f}ms")
print(f"   Hybrid (BM25+Dense):      {latency_hybrid:.0f}ms (+{latency_hybrid/latency_dense:.1f}x)")
print(f"   Balanced (Dense→Rerank):   {latency_rerank:.0f}ms (+{latency_rerank/latency_dense:.1f}x)")
print(f"   Quality (Multi→Hybrid→Rerank): {latency_mqh:.0f}ms (+{latency_mqh/latency_dense:.1f}x)")
print("\n💡 Trade-off: Higher latency → Better quality")
print("   Pipelines 3 & 4 are sorted by rerank scores (most precise!)")
print("   Choose based on your speed vs quality requirements!")
print("=" * 80)

---

# 📊 Summary & Key Takeaways

## What You've Learned

### From 2A: Core Techniques (Building Blocks)
1. 🟢 Dense Search - Semantic similarity baseline
2. 🟡 Hybrid Search - BM25 + Dense + RRF fusion
3. 🟠 Reranking - Two-stage precision boost
4. 🔵 Parent-Child - Adaptive context sizing

### From 2B: Advanced Orchestration (Smart Systems)
5. 🔄 Multi-Query - Variant generation + fusion (+15-20%)
6. 🎯 Query Routing - Auto-select method by type (+15-20%)
7. 🧠 Adaptive Retrieval - Auto-configure by complexity (+20-30% efficiency)

---

## Decision Framework

### Start Here (Beginner)
```
Hybrid Search (from 2A)
  ↓
Good baseline for most applications!
~100ms latency, +15-25% over Dense
```

### Level Up (Intermediate)
```
Add ONE orchestration technique:
  • Query Routing (if varied query types)
  • Adaptive Retrieval (if varied complexity)
  • Multi-Query (if need breadth)
```

### Production Ready (Advanced)
```
Combine techniques into pipeline:
  • Balanced: Routing + Hybrid + Rerank
  • Quality: Multi-Query + Hybrid + Rerank  
  • Smart: Routing + Adaptive
```

---

## Quick Reference Table

| Technique | Use When | Gain | Latency | From |
|-----------|----------|------|---------|------|
| Dense | Baseline, simple queries | - | ~20ms | 2A |
| Hybrid | Need keyword matching | +15-25% | ~100ms | 2A |
| Reranking | Need precision | +10-15% | +50ms | 2A |
| Parent-Child | Need context | Varies | ~20ms | 2A |
| Multi-Query | Complex/comparison | +15-20% | +200ms | 2B |
| Routing | Varied query types | +15-20% | 0ms | 2B |
| Adaptive | Varied complexity | +30% eff | Auto | 2B |

**Combinations:**
- Hybrid + Rerank: +30-40% (balanced)
- Multi-Query + Hybrid + Rerank: +45-60% (maximum quality)
- Routing + Adaptive: +30-50% quality, +40% efficiency (smart)

---

## Common Patterns

**Pattern 1: Progressive Enhancement**
```
Start: Dense (baseline)
  ↓ (not good enough?)
Add: Hybrid Search
  ↓ (still not good enough?)
Add: Reranking
  ↓ (need more coverage?)
Add: Multi-Query
```

**Pattern 2: Smart from Start**
```
Start: Routing + Adaptive
  ↓
System auto-selects best method per query
  ↓
Adjust thresholds based on metrics
```

**Pattern 3: Domain-Specific**
```
Technical docs: Hybrid (keyword matching important)
Medical/Legal: Multi-Query + Rerank (precision critical)
Chatbot: Routing + Adaptive (efficiency + quality)
Research: Multi-Query + Parent-Child (breadth + context)
```

---

## Next Steps

1. **Experiment** with these techniques on your own data
2. **Measure** performance (latency, quality, cost)
3. **Choose** a starting pipeline from the 4 options above
4. **Iterate** based on your specific requirements
5. **Monitor** in production and adjust as needed

---

## Pro Tips 💡

1. **Always measure** - Don't assume complex = better
2. **Start simple** - Add complexity only when needed
3. **Combine wisely** - More techniques ≠ always better
4. **Monitor latency** - User experience matters
5. **A/B test** - Validate improvements with real queries
6. **Consider cost** - Multi-Query = 4x searches = 4x cost
7. **Use routing** - Skip retrieval when not needed
8. **Cache results** - Same query? Don't search again

---

## Resources

- **Code:** `retrieval_playground/src/mid_retrieval/`
- **Tests:** `retrieval_playground/tests/`
- **Docs:** `MID_RETRIEVAL_OPTIMIZATION_PLAN.md`

---

# 🎉 Congratulations!

You've completed the Mid-Retrieval workshop!

You now know:
- ✅ 4 core retrieval techniques (2A)
- ✅ 3 advanced orchestration methods (2B)
- ✅ How to combine them into production pipelines
- ✅ When to use which approach

**You're ready to build production RAG systems!** 🚀

Next: Post-Retrieval (coming soon) - Context compression, answer generation, and more!